# Poster polar plot

Set `INPUT_PATH` in the next cell. It can be either:

- the project `config.toml`, for example `C:/.../my_project/config.toml`
- the analysis output `feature_df.parquet`, for example `C:/.../my_project/results/feature_df.parquet`

If you use `config.toml`, the notebook looks for `results/feature_df.parquet` inside that project.

In [ ]:
# Change this to your project config.toml or feature_df.parquet
INPUT_PATH = r"C:\Repositories\DeepEcoHab\examples\test_data\test_2025-12-29\config.toml"

# Filters and output options
PHASES = ["dark_phase", "light_phase"]  # or ["dark_phase"] / ["light_phase"]
FIRST_DAY = None  # e.g. 1
LAST_DAY = None   # e.g. 6
TITLE = "Animal Feature Overview"
OUTPUT_PREFIX = r"poster_polar_plot"  # no extension

# Poster sizing. SVG/PDF are best for posters because they stay sharp.
WIDTH = 1500
HEIGHT = 1200
EXPORT_HTML = True
EXPORT_SVG = True
EXPORT_PNG = False
EXPORT_PDF = False
STATIC_SCALE = 3

In [ ]:
from pathlib import Path
import math

import plotly.graph_objects as go
import plotly.express as px
import polars as pl
import toml

POSTER_TEMPLATE = "plotly_white"
METRIC_LABELS = {
    "time_alone": "Time Alone",
    "n_chasing": "Chasing Initiated",
    "n_chased": "Chasing Received",
    "n_wins": "Tube-test Wins",
    "n_loses": "Tube-test Losses",
    "activity": "Activity",
    "time_together": "Time Together",
    "pairwise_encounters": "Pairwise Encounters",
}

def color_sampling(values, cmap="Phase"):
    x = [i / len(values) for i in range(len(values) + 1)]
    return px.colors.sample_colorscale(cmap, x)

def resolve_feature_df(input_path):
    input_path = Path(input_path)
    if input_path.suffix == ".parquet":
        return pl.read_parquet(input_path), {}

    cfg = toml.load(input_path)
    feature_path = Path(cfg["project_location"]) / "results" / "feature_df.parquet"
    if not feature_path.exists():
        raise FileNotFoundError(f"Could not find {feature_path}. Run the analysis first.")
    return pl.read_parquet(feature_path), cfg

def default_day_range(feature_df, first_day=None, last_day=None):
    first = first_day if first_day is not None else int(feature_df["day"].min())
    last = last_day if last_day is not None else int(feature_df["day"].max())
    return [first, last]

def prepare_polar_df(feature_df, days_range, phases, animal_order=None):
    n_days = 1 if days_range[0] == days_range[1] else days_range[1] - days_range[0] + 1
    df = (
        feature_df.lazy()
        .filter(
            pl.col("phase").is_in(phases),
            pl.col("day").is_between(days_range[0], days_range[1]),
        )
        .group_by("animal_id", "metric", "day")
        .agg(pl.mean("z-score"))
        .group_by("animal_id", "metric")
        .agg(
            pl.mean("z-score").alias("mean"),
            (pl.std("z-score") / math.sqrt(n_days)).alias("sem"),
        )
        .with_columns(
            (pl.col("mean") - pl.col("sem")).alias("lower"),
            (pl.col("mean") + pl.col("sem")).alias("upper"),
            pl.col("animal_id").cast(pl.String),
        )
        .fill_null(0)
    )

    if animal_order:
        order_map = {animal: idx for idx, animal in enumerate(animal_order)}
        df = df.with_columns(
            pl.col("animal_id").replace(order_map, default=len(order_map)).cast(pl.Int64).alias("_animal_order")
        )
        out = df.sort("_animal_order", "metric").drop("_animal_order").collect(engine="in-memory")
    else:
        out = df.sort("animal_id", "metric").collect(engine="in-memory")

    return out.with_columns(
        pl.col("metric").replace(METRIC_LABELS).str.to_titlecase().str.replace("_", " ").alias("metric")
    )

In [ ]:
def poster_polar_plot(df, animals, title, width, height):
    colors = color_sampling(animals)
    color_map = dict(zip(animals, colors))
    fig = go.Figure()

    for animal, group in df.partition_by("animal_id", as_dict=True).items():
        animal_id = str(animal[0] if isinstance(animal, tuple) else animal)
        group_closed = pl.concat([group, group.head(1)])
        color = color_map.get(animal_id, "#2F5597")
        shade_color = color.replace("rgb", "rgba").replace(")", ", 0.16)")

        fig.add_trace(go.Scatterpolar(
            r=group_closed["lower"], theta=group_closed["metric"], mode="lines",
            line={"width": 0, "color": color}, legendgroup=animal_id,
            showlegend=False, hoverinfo="skip", name=f"{animal_id} lower",
        ))
        fig.add_trace(go.Scatterpolar(
            r=group_closed["upper"], theta=group_closed["metric"], mode="lines",
            fill="tonext", fillcolor=shade_color, line={"width": 0, "color": color},
            legendgroup=animal_id, showlegend=False, hoverinfo="skip", name=f"{animal_id} SEM",
        ))
        fig.add_trace(go.Scatterpolar(
            r=group_closed["mean"], theta=group_closed["metric"], mode="lines+markers",
            line={"color": color, "width": 5},
            marker={"size": 10, "color": color, "line": {"color": "white", "width": 1.5}},
            legendgroup=animal_id, name=animal_id,
            hovertemplate="Animal: %{fullData.name}<br>Metric: %{theta}<br>Z-score: %{r:.2f}<extra></extra>",
        ))

    r_min = float(df["lower"].min()) - 0.35
    r_max = float(df["upper"].max()) + 0.35
    fig.update_layout(
        template=POSTER_TEMPLATE,
        title={"text": f"<b>{title}</b>", "x": 0.5, "xanchor": "center", "font": {"size": 38}},
        width=width, height=height,
        font={"family": "Arial", "size": 24, "color": "#1A1A1A"},
        paper_bgcolor="white", plot_bgcolor="white",
        legend={
            "title": {"text": "<b>Animal ID</b>", "font": {"size": 25}},
            "font": {"size": 20}, "orientation": "v",
            "x": 1.04, "y": 0.5, "xanchor": "left", "yanchor": "middle",
            "itemsizing": "constant", "tracegroupgap": 4,
        },
        margin={"l": 95, "r": 260, "t": 115, "b": 90},
        polar={
            "bgcolor": "white",
            "radialaxis": {"visible": True, "range": [r_min, r_max], "gridcolor": "rgba(0,0,0,0.16)", "gridwidth": 2, "linecolor": "rgba(0,0,0,0.35)", "linewidth": 2, "tickfont": {"size": 19}},
            "angularaxis": {"gridcolor": "rgba(0,0,0,0.13)", "gridwidth": 2, "linecolor": "rgba(0,0,0,0.35)", "linewidth": 2, "tickfont": {"size": 22}, "rotation": 90, "direction": "clockwise"},
        },
    )
    return fig

In [ ]:
feature_df, cfg = resolve_feature_df(INPUT_PATH)
days_range = default_day_range(feature_df, FIRST_DAY, LAST_DAY)
animals = [str(animal) for animal in cfg.get("animal_ids", [])]
if not animals:
    animals = feature_df["animal_id"].cast(pl.String).unique().sort().to_list()

polar_df = prepare_polar_df(feature_df, days_range, PHASES, animals)
fig = poster_polar_plot(polar_df, animals, TITLE, WIDTH, HEIGHT)
fig

In [ ]:
output_prefix = Path(OUTPUT_PREFIX)
output_prefix.parent.mkdir(parents=True, exist_ok=True)

if EXPORT_HTML:
    fig.write_html(output_prefix.with_suffix(".html"))
if EXPORT_SVG:
    fig.write_image(output_prefix.with_suffix(".svg"), scale=STATIC_SCALE)
if EXPORT_PNG:
    fig.write_image(output_prefix.with_suffix(".png"), scale=STATIC_SCALE)
if EXPORT_PDF:
    fig.write_image(output_prefix.with_suffix(".pdf"), scale=STATIC_SCALE)

print(f"Saved outputs with prefix: {output_prefix.resolve()}")